In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("brain_tumor_risk_synthetic_dataset.csv")

In [4]:
df.head()

,age,gender,family_history,persistent_headache,worsening_headache,morning_headache,nausea,vomiting,seizures,vision_problems,...,sleep_changes,mood_changes,loss_of_consciousness,hormonal_changes,symptom_count,neurological_symptom_score,cognitive_symptom_score,risk_score,brain_tumor_risk_level,brain_tumor_risk_percentage
0,56,Female,0,0,0,0,0,0,1,0.0,...,0.0,0.0,0.0,1.0,4.0,4.0,0.0,34.9,1.0,34.9
1,19,Male,0,0,0,0,0,0,0,1.0,...,0.0,0.0,0.0,0.0,4.0,10.0,0.0,49.3,1.0,49.3
2,76,Female,0,0,0,0,1,0,0,0.0,...,0.0,0.0,0.0,0.0,4.0,9.0,0.0,41.4,1.0,41.4
3,65,Female,0,0,0,0,0,0,0,1.0,...,0.0,1.0,1.0,0.0,6.0,14.0,3.0,96.2,2.0,96.2
4,25,Female,0,0,0,0,0,0,0,0.0,...,0.0,0.0,0.0,0.0,2.0,4.0,2.0,29.8,0.0,29.8


In [5]:
df.shape

(12378, 37)

In [6]:
df.columns

Index(['age', 'gender', 'family_history', 'persistent_headache',
       'worsening_headache', 'morning_headache', 'nausea', 'vomiting',
       'seizures', 'vision_problems', 'balance_problems', 'one_side_weakness',
       'numbness', 'memory_loss', 'confusion', 'concentration_difficulty',
       'personality_changes', 'speech_difficulty',
       'language_understanding_difficulty', 'mental_fog', 'hearing_problems',
       'tinnitus', 'loss_of_smell', 'facial_weakness', 'difficulty_swallowing',
       'poor_hand_coordination', 'extreme_fatigue', 'sleep_changes',
       'mood_changes', 'loss_of_consciousness', 'hormonal_changes',
       'symptom_count', 'neurological_symptom_score',
       'cognitive_symptom_score', 'risk_score', 'brain_tumor_risk_level',
       'brain_tumor_risk_percentage'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12378 entries, 0 to 12377
Data columns (total 37 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   age                                12378 non-null  int64  
 1   gender                             12378 non-null  object 
 2   family_history                     12378 non-null  int64  
 3   persistent_headache                12378 non-null  int64  
 4   worsening_headache                 12378 non-null  int64  
 5   morning_headache                   12378 non-null  int64  
 6   nausea                             12378 non-null  int64  
 7   vomiting                           12378 non-null  int64  
 8   seizures                           12378 non-null  int64  
 9   vision_problems                    12377 non-null  float64
 10  balance_problems                   12377 non-null  float64
 11  one_side_weakness                  12377 non-null  flo

In [8]:
df.isnull().sum()

,0
age,0
gender,0
family_history,0
persistent_headache,0
worsening_headache,0
morning_headache,0
nausea,0
vomiting,0
seizures,0
vision_problems,1


In [9]:
df = df.dropna()

In [10]:
df.isnull().sum().sum()

np.int64(0)

In [11]:
# Target distribution
df['brain_tumor_risk_level'].value_counts()

,count
brain_tumor_risk_level,
0.0,6416
1.0,4455
2.0,1506


In [12]:
# Percentage distribution
df['brain_tumor_risk_level'].value_counts(normalize=True) * 100

,proportion
brain_tumor_risk_level,
0.0,51.838087
1.0,35.994183
2.0,12.167730


| Class | Meaning (assumed) | %         |
| ----- | ----------------- | --------- |
| 0     | Low Risk          | **51.8%** |
| 1     | Medium Risk       | **36.0%** |
| 2     | High Risk         | **12.2%** |

# Feature Selection

In [13]:
X = df.drop(['brain_tumor_risk_level','risk_score','brain_tumor_risk_percentage'], axis=1)
y = df['brain_tumor_risk_level']

In [14]:
X.shape

(12377, 34)

In [15]:
# Categorical columns
X.select_dtypes(include='object').columns

Index(['gender'], dtype='object')

In [20]:
# Numerical columns
num_cols = X.select_dtypes(exclude='object').columns
num_cols

Index(['age', 'family_history', 'persistent_headache', 'worsening_headache',
       'morning_headache', 'nausea', 'vomiting', 'seizures', 'vision_problems',
       'balance_problems', 'one_side_weakness', 'numbness', 'memory_loss',
       'confusion', 'concentration_difficulty', 'personality_changes',
       'speech_difficulty', 'language_understanding_difficulty', 'mental_fog',
       'hearing_problems', 'tinnitus', 'loss_of_smell', 'facial_weakness',
       'difficulty_swallowing', 'poor_hand_coordination', 'extreme_fatigue',
       'sleep_changes', 'mood_changes', 'loss_of_consciousness',
       'hormonal_changes', 'symptom_count', 'neurological_symptom_score',
       'cognitive_symptom_score'],
      dtype='object')

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [22]:
# Preprocessing define

# Numerical preprocessing
num_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])

In [23]:
# Categorical preprocessing
cat_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))])

In [24]:
# Combine pipelines
preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols),('cat', cat_pipeline, ['gender'])])

In [25]:
# Train–Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [26]:
X_train.shape, X_test.shape

((9901, 34), (2476, 34))

In [27]:
# Model Training Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [28]:
model = Pipeline(steps=[('preprocessing', preprocessor),('classifier', LogisticRegression(max_iter=1000,multi_class='auto',n_jobs=-1))])

In [29]:
# Model train
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'family_history', 'persistent_headache', 'worsening_headache',
       'morning_headache', 'nausea', 'vomiting', 'seizures', 'vision_problems',
       'balance_problems', 'one_side_weakness', 'numbne...
       'sleep_changes', 'mood_changes', 'loss_of_consciousness',
       'hormonal_changes', 'symptom_count', 'neurological_symptom_score',
       'cognitive_symptom_score'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, multi_class='auto',
                                    n_jobs=-1))])

In [30]:
y_pred = model.predict(X_test)

In [31]:
y_train_pred = model.predict(X_train)

In [32]:
# Train Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
train_accuracy

0.901020098979901

In [33]:
# Test Accuracy
accuracy_score(y_test, y_pred)

0.9079159935379645

In [34]:
# Classification Report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94      1284
         1.0       0.86      0.89      0.87       891
         2.0       0.90      0.84      0.87       301

    accuracy                           0.91      2476
   macro avg       0.90      0.89      0.90      2476
weighted avg       0.91      0.91      0.91      2476



In [35]:
# Confusion Matrix
confusion_matrix(y_test, y_pred)

array([[1204,   80,    0],
       [  73,  790,   28],
       [   0,   47,  254]])